In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Deep Learning Models/RNN Sentiment Analysis/IMDB Dataset.csv')

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.shape

(50000, 2)

In [5]:
df.isnull().sum()

,0
review,0
sentiment,0


In [6]:
df.drop_duplicates(inplace=True)
df.shape

(49582, 2)

**Pre - Processing**
1. Converting to lowercase

In [7]:
df["review"] = df["review"].str.lower()

2. Removing the URLs

In [8]:
import re


In [9]:
def remove_urls(text):
  text = re.sub(r"http\S+", "", text) #pattern,replace,string
  return text

df["review"] = df["review"].apply(remove_urls)

3. Removing Punctuations

In [10]:
def remove_punctuations(text):
  text = re.sub(r"^A-Za-z0-9\s]", "", text)
  return text
df["review"] = df["review"].apply(remove_punctuations)

4.Removing HTML Tags

In [11]:
def remove_html(text):
  text = re.sub(r"<.*?>", "", text)
  return text
df["review"] = df["review"].apply(remove_html)

5. Removing the StopWords

In [12]:
import nltk #ntlk = natural language toolkit
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [13]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [14]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
          text = text.replace(word, "")
    return text

df["review"] = df["review"].apply(remove_stopwords)

In [15]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode 'll h...,positive
1,wderful ltle producti. filming technique u...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly 's fmly lttle boy (jke) thks 's zom...,negative
4,"petter mttei's ""love time mey"" vully stun...",positive


6. Stemming

In [16]:
# Porter Stemming
from nltk.stem import PorterStemmer
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)
df["review"] = df["review"].apply(stemming)


In [17]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod 'll hook . y rght...,positive
1,wder ltle producti . film techniqu unssuming- ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli 's fmli lttle boy ( jke ) thk 's zomb c...,negative
4,petter mttei 's `` love time mey '' vulli stun...,positive


7.Encoding

In [18]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

In [19]:
y = df["sentiment"]
y

,sentiment
0,1
1,1
2,1
3,0
4,1
...,...
49995,1
49996,0
49997,0
49998,0


8.Vectorization

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4093289 stored elements and shape (49582, 5000)>

**Dataset & Data Loaders**

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [22]:
X_train.shape

(39665, 5000)

In [23]:
X_test.shape

(9917, 5000)

In [24]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train = X_train.toarray()
X_test = X_test.toarray()

train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [25]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

Build Our RNN

In [30]:
#many to one RNN architecture
import torch.nn as nn
import torch.optim as optim
class RNN(nn.Module):
  def __init__(self, input_size, hidden_size=128, num_layers=1):
    super().__init__()

    self.hidden_size = hidden_size
    self.num_layers = num_layers

    #RNN layer
    self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

    #Fully Connected layer
    self.fc = nn.Linear(hidden_size, 1)

    #fp
  def forward(self, x):
    h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

    out, _ = self.rnn(x, h0)
#1st value = hidden state of all the timesteps =>(batch, seq_len, hidden size)
#2nd value = final hidden state of last timestep

    out = self.fc(out[:, -1, :])
    return out

In [32]:
input_size = X_train.shape[1]
model = RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

Training the RNN

In [33]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction

        outputs = model(Xb) # (batch_size,1)

        outputs = torch.sigmoid(outputs.squeeze()) #(batch_size,)=probability

        loss = criterion(outputs, yb) #compute loss
        loss.backward() #bp
        optimizer.step() #weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.3380817472934723
epoch = 2/10 and loss = 0.31440189480781555
epoch = 3/10 and loss = 0.1816217303276062
epoch = 4/10 and loss = 0.19789519906044006
epoch = 5/10 and loss = 0.2291664183139801
epoch = 6/10 and loss = 0.2680622339248657
epoch = 7/10 and loss = 0.2414201945066452
epoch = 8/10 and loss = 0.2489360123872757
epoch = 9/10 and loss = 0.24905212223529816
epoch = 10/10 and loss = 0.3346155881881714


In [34]:
# Evaluate

model.eval()
with torch.no_grad():
     correct_vals = 0
     tot_vals = 0

     for Xb, yb in test_loader:
         Xb = Xb.unsqueeze(1)

         outputs = model(Xb)
         predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

         tot_vals += yb.size(0)
         correct_vals += (predicted == yb).sum().item()
     print(f"accuracy = {correct_vals/tot_vals*100} ")

accuracy = 85.6206514066754 
